# 5.1

Подключим необходимые библиотеки

In [16]:
import numpy as np
from itertools import combinations, product

Функция для генерации строки порождающей матрицы

In [17]:
def generate_row(bit_reversed_vectors, subset):
    """Генерирует строку матрицы на основе подмножества индексов"""
    row = [(np.prod([(x + 1) for i, x in enumerate(vector) if i in subset]) % 2) for vector in bit_reversed_vectors]
    return row

Функция для сортировки порождающей матрицы

In [18]:
def sort_rows_by_seniority(matrix, subset_lengths):
    """Сортирует строки матрицы в каждой группе подмножеств по старшему отличающемуся биту"""
    def sort_key(row):
        # Находим индекс старшего отличающегося бита (с конца) и его значение
        for idx in range(len(row) - 1, -1, -1):
            if row[idx] == 1:
                return (idx, row[idx])
        return (len(row), 0)  # Для строк, состоящих только из нулей
    
    # Сортировка внутри каждой группы подмножеств
    sorted_matrix = []
    start = 0
    for length in subset_lengths:
        end = start + length
        sorted_matrix.extend(sorted(matrix[start:end], key=sort_key))
        start = end

    return np.array(sorted_matrix, dtype=int)

Функция для создания всей порождающей матрицы

In [19]:
def reed_muller_generator_matrix(r, m):
    """Создает порождающую матрицу для кода Рида-Маллера с параметрами r и m"""
    bit_reversed_vectors = [list(map(int, f"{i:0{m}b}"[::-1])) for i in range(2 ** m)]
    
    # Генерация всех подмножеств и их длины
    all_subsets = []
    subset_lengths = []
    for i in range(r + 1):
        subsets = list(combinations(range(m), i))
        all_subsets.extend(subsets)
        subset_lengths.append(len(subsets))
    
    # Создание матрицы без сортировки
    matrix = np.array([generate_row(bit_reversed_vectors, subset) for subset in all_subsets], dtype=int)
    
    # Сортировка матрицы по старшему отличающемуся биту
    return sort_rows_by_seniority(matrix, subset_lengths)


Построим порождающую матрицу кода Рида-Маллера в наноническом виде с параметрами r=3, m=4

In [20]:
r, m = 2, 4
G = reed_muller_generator_matrix(r, m)
print(G)

[[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0]
 [1 1 1 1 0 0 0 0 1 1 1 1 0 0 0 0]
 [1 1 0 0 1 1 0 0 1 1 0 0 1 1 0 0]
 [1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0]
 [1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 0 0 1 1 0 0 0 0 0 0 0 0 0 0]
 [1 0 1 0 1 0 1 0 0 0 0 0 0 0 0 0]
 [1 1 0 0 0 0 0 0 1 1 0 0 0 0 0 0]
 [1 0 1 0 0 0 0 0 1 0 1 0 0 0 0 0]
 [1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0]]


# 5.2

In [21]:
# Сортировка для декодирования
def sort_for_major(m, r):
    indices = range(m)

    combinations_list = list(combinations(indices, r))
    if combinations_list:
        combinations_list.sort(key=lambda x: len(x))

    result = np.array(combinations_list, dtype=int)
    return result

In [22]:
# Вычисляем значение функции f для множества I
def compute_f_value(vector, subset):
    return np.prod([(vector[idx] + 1) % 2 for idx in subset])

In [23]:
# Формируем бинарную матрицу с 2^cols строк и cols столбцами (все возможные двоичные векторы длины cols)
def generate_binary_matrix(cols):
    return list(product([0, 1], repeat=cols))

In [24]:
# Формируем вектор V_I на основе множества I
def get_vector_V(subset, num_cols):
    if len(subset) == 0:
        return np.ones(2 ** num_cols, int)
    else:
        v_vector = []
        for binary_vector in generate_binary_matrix(num_cols):
            f_value = compute_f_value(binary_vector, subset)
            v_vector.append(f_value)
        return v_vector

In [25]:
def get_vector_H(I, m):
    return [word for word in generate_binary_matrix(m) if compute_f_value(word, I) == 1]

In [26]:
# Формирование комплиментарного множества
def get_Komplement(I, m):
    return [i for i in range(m) if i not in I]

In [27]:
def compute_f_t_value(words, I, t):
    return np.prod([(words[j] + t[j] + 1) % 2 for j in I])

In [28]:
def get_V_I_t(I, m, t):
    if not I:
        return np.ones(2 ** m, dtype=int)
    return [compute_f_t_value(word, I, t) for word in generate_binary_matrix(m)]

In [29]:
# Мажоритарное декодирование
def major_algorithm(w, r, m, size):
    i = r
    w_r = w.copy()
    Mi = np.zeros(size, dtype=int)
    max_weight = 2**(m - r - 1) - 1
    index = 0

    while True:
        for J in sort_for_major(m, i):
            max_zeros_and_ones_count = 2**(m - i - 1)
            zeros_count = 0
            ones_count = 0
            for t in get_vector_H(J, m):
                komplement = get_Komplement(J, m)
                V = get_V_I_t(komplement, m, t)
                c = np.dot(w_r, V) % 2

                if c == 0:
                    zeros_count += 1
                else:  # c == 1
                    ones_count += 1

            if zeros_count > max_weight and ones_count > max_weight:
                return
            if zeros_count > max_zeros_and_ones_count:
                Mi[index] = 0
                index += 1
            if ones_count > max_zeros_and_ones_count:
                Mi[index] = 1
                index += 1
                V = get_vector_V(J, m)
                w_r = (w_r + V) % 2

        if i > 0:
            if len(w_r) < max_weight:
                for J in sort_for_major(m, r + 1):
                    Mi[index] = 0
                    index += 1
                break
            i -= 1
        else:
            break

    reversed(Mi)
    return Mi

In [30]:
# Генерация слова с указанным количеством ошибок
def generate_word_with_n_mistakes(G, error_count):
    u = np.array([1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1])
    print("Исходное сообщение: ", u)

    u = u.dot(G) % 2
    mistake_pos = np.random.choice(len(u), size=error_count, replace=False)
    u[mistake_pos] = (u[mistake_pos] + 1) % 2

    return u

# 5.3

In [31]:
# Эксперимент для однократной ошибки
Err = generate_word_with_n_mistakes(G, 1)
print("Слово с однократной ошибкой:", Err)
print()

Decoded_word = major_algorithm(Err, 2, 4, len(G))
if Decoded_word is None:
    print("\nНеобходима повторная отправка сообщения")
else:
    print("Исправленное слово:", Decoded_word)
    V2 = Decoded_word.dot(G) % 2
    print("Результат умножения исправленного слова на матрицу G:", V2)

Исходное сообщение:  [1 0 0 0 1 1 0 0 0 1 1]
Слово с однократной ошибкой: [1 0 0 0 1 1 0 0 0 1 1 1 1 1 0 1]

Исправленное слово: [1 0 0 0 1 1 0 0 0 1 1]
Результат умножения исправленного слова на матрицу G: [1 0 0 0 1 1 0 1 0 1 1 1 1 1 0 1]


In [32]:
# Эксперимент для двукратной ошибки
Err = generate_word_with_n_mistakes(G, 2)
print("Слово с двукратной ошибкой:", Err)

Decoded_word = major_algorithm(Err, 2, 4, len(G))
if Decoded_word is None:
   print("\nНеобходима повторная отправка сообщения")
else:
   print("Исправленное слово:", Decoded_word)
   V2 = Decoded_word.dot(G) % 2
   print("Результат умножения исправленного слова на матрицу G:", V2)

Исходное сообщение:  [1 0 0 0 1 1 0 0 0 1 1]
Слово с двукратной ошибкой: [1 0 0 0 0 1 0 1 0 1 1 1 1 0 0 1]

Необходима повторная отправка сообщения
